## Load all data and extraction of data to run RTCD in R
    ### 1. All cells from Christoffer (data has to be NOT normalized)
    ### 2. Cells from Banksy object (to subset the previous object with the cells of interest)

In [ ]:
import scanpy as sc
import anndata as ad

adata1 = sc.read_h5ad("/Users/anajul/Desktop/Goritz_Lab/Spatial_transcriptomic/All_daasets_14082025/combined_m_50_s_4_outer_join_raw_data.h5ad")
adata2 = sc.read_h5ad("/Users/anajul/Desktop/Goritz_Lab/Spatial_transcriptomic/Analyse_3/Crush_banksy_integration_with_matrix.h5ad")

# Make sure adata1 and adata2 share cell identifiers
cells_to_keep = adata1.obs_names.intersection(adata2.obs_names)

# Subset adata1
adata = adata1[cells_to_keep].copy()

# Add the "manual_anno" column from adata2.obs to the new object
adata.obs["manual_anno"] = adata2.obs.loc[cells_to_keep, "manual_anno"]


In [ ]:
print(adata)

In [ ]:
adata
adata.obs.head()

In [ ]:
adata
adata.X.shape

In [ ]:
## Split the obejct by slide (otherwise the coordinates overlap) 

import os
import re

# Get unique slide IDs from metadata
slide_ids = adata.obs["slide"].unique()

# Dictionary of AnnData objects, one per slide
slide_datasets = {
    slide: adata[adata.obs["slide"] == slide].copy()
    for slide in slide_ids
}

# Define your output folder
output_folder = "/Users/anajul/Desktop/Goritz_Lab/Spatial_transcriptomic/Analyse_3/For_RTCD"
os.makedirs(output_folder, exist_ok=True)

# Save each slide-specific AnnData object
for slide, adata_slide in slide_datasets.items():
    # Clean the slide name to make a safe filename
    safe_slide = re.sub(r"[^a-zA-Z0-9_-]", "_", str(slide))
    filename = os.path.join(output_folder, f"{safe_slide}.h5ad")
    adata_slide.write(filename)


In [ ]:
adata = sc.read_h5ad("/Users/anajul/Desktop/Goritz_Lab/Spatial_transcriptomic/Analyse_3/For_RTCD/Slide_12.h5ad")

In [ ]:
import numpy as np

x = adata.obs["x"]
y = adata.obs["y"]

adata.obsm["spatial"] = np.vstack([adata.obs["x"], adata.obs["y"]]).T

coord_keys = ('x', 'y', 'spatial')


In [ ]:
print(adata)

In [ ]:
## For each oject per slide, keep only raw counts and spatial coordinates (otherwise the conversion to R does not work)

import pandas as pd

# Keep only counts
adata.obs = pd.DataFrame(index=adata.obs.index)  # keep cell IDs
adata.var = pd.DataFrame(index=adata.var.index)  # keep gene names

# Keep spatial coordinates if they exist
# Replace 'spatial' with the actual key in obsm if different
if 'spatial' in adata.obsm:
    spatial_coords = adata.obsm['spatial']
    adata.obsm = {'spatial': spatial_coords}
else:
    adata.obsm = {}

# Remove other metadata
adata.uns = {}
adata.varm = {}
adata.layers = {}


# Save cleaned file
adata.write_h5ad("/Users/anajul/Desktop/Goritz_Lab/Spatial_transcriptomic/Analyse_3/For_RTCD/forR_Slide_12.h5ad")


In [ ]:
## Extract the data needed to create the object in R

import scanpy as sc
import pandas as pd
import numpy as np

# 1️⃣ Raw counts
# If X is already raw counts
counts = adata.X.copy()  # shape (n_obs, n_vars)

# Convert to DataFrame
counts_df = pd.DataFrame(counts.toarray() if hasattr(counts, "toarray") else counts,
                         index=adata.obs_names,
                         columns=adata.var_names)

counts = adata.X.toarray() if hasattr(adata.X, "toarray") else adata.X
counts_df = pd.DataFrame(counts, index=adata.obs_names, columns=adata.var_names)
counts_df.to_csv("counts_numeric.csv", float_format="%.0f")  # ensure integer counts

# 2️⃣ Coordinates
coords = pd.DataFrame(adata.obsm['spatial'], index=adata.obs_names, columns=['x','y'])
coords['cell'] = coords.index

counts_df.to_csv("/Users/anajul/Desktop/Goritz_Lab/Spatial_transcriptomic/Analyse_3/For_RTCD/Slide_12_counts.csv")  
coords.to_csv("/Users/anajul/Desktop/Goritz_Lab/Spatial_transcriptomic/Analyse_3/For_RTCD/Slide_12_coords.csv")



## Analyses of RCTD results (generated in R)

In [ ]:
import pandas as pd
import scanpy as sc
import anndata as ad

# -----------------------------
# Step 1: Load your AnnData object
# -----------------------------

adata = sc.read_h5ad("/Users/anajul/Desktop/Goritz_Lab/Spatial_transcriptomic/Analyse_3/Crush_uninjured_nonspatial.h5ad")
# -----------------------------
# Step 2: Load the RTCD assignments
# -----------------------------

anno_RTCD = pd.read_csv("~/Desktop/Goritz_Lab/Spatial_transcriptomic/Analyse_3/For_RTCD/annotations_df.csv", sep=";", index_col=0)

# Clean column names (remove quotes if present)
anno_RTCD.columns = [c.strip('"') for c in anno_RTCD.columns]

# Give the index a name for clarity
anno_RTCD.index.name = "cell_id"

# -----------------------------
# Step 3: Initialize assignments
# -----------------------------
# All cells start as "not_mapped"
cell_assignments = pd.Series("not_mapped", index=adata.obs_names)

# -----------------------------
# Step 4: Fill in mapped cells
# -----------------------------
# Replace 'first_type' with the actual column in your CSV that contains RTCD assignments
cell_assignments.loc[anno_RTCD.index] = anno_RTCD["first_type"]

# -----------------------------
# Step 5: Add to adata.obs
# -----------------------------
adata.obs["rtcd_assignment"] = cell_assignments

# -----------------------------
# Step 6: Save the updated AnnData object (optional)
# -----------------------------
adata.write_h5ad("/Users/anajul/Desktop/Goritz_Lab/Spatial_transcriptomic/Analyse_3/For_RTCD/Crush_uninjured_nonspatial_RTCD.h5ad")

# -----------------------------
# Done!
# -----------------------------
print("RTCD assignments added. Unmapped cells labeled 'not_mapped'.")
print(adata.obs["rtcd_assignment"].value_counts())


In [ ]:
print(adata)

In [ ]:
sc.pl.umap(
    adata, 
    color='rtcd_assignment', 
    legend_loc='right margin'  # or 'best', 'upper right', etc.
)

In [ ]:
# Subset to only slide_12
adata_slide12 = adata[adata.obs['slide'] == 'Slide#12', :]

# Plot UMAP only for that subset
sc.pl.umap(
    adata_slide12, 
    color='rtcd_assignment', 
    legend_loc='right margin'
)


In [ ]:
rtcd_assignment = adata.obs['rtcd_assignment'].unique()

for s in rtcd_assignment:
    sc.pl.umap(
        adata[adata.obs['rtcd_assignment'] == s],
        color='rtcd_assignment',
        legend_loc='on data',
        title=f"UMAP - {s}",
    )

In [ ]:
adata = sc.read_h5ad("/Users/anajul/Desktop/Goritz_Lab/Spatial_transcriptomic/Analyse_3/Slide12_all.h5ad")

## Analyses results of RCTD for slide 12 only 

In [ ]:
import scanpy as sc
import anndata as ad

adata1 = sc.read_h5ad("/Users/anajul/Desktop/Goritz_Lab/Spatial_transcriptomic/Analyse_3/Crush_uninjured_nonspatial.h5ad")
adata2 = sc.read_h5ad("/Users/anajul/Desktop/Goritz_Lab/Spatial_transcriptomic/Analyse_3/For_RTCD/forR_Slide_12.h5ad")

# Make sure adata1 and adata2 share cell identifiers
cells_to_keep = adata1.obs_names.intersection(adata2.obs_names)

# Subset adata1
adata = adata1[cells_to_keep].copy()


In [ ]:
print(adata)

In [ ]:
# Add RTCD annotations as metadata in python object

anno_RTCD = pd.read_csv("~/Desktop/Goritz_Lab/Spatial_transcriptomic/Analyse_3/For_RTCD/annotations_df.csv", sep=";", index_col=0)
adata.obs = adata.obs.join(anno_RTCD)

In [ ]:
# Add weights as metadata in python object

weights = pd.read_csv("~/Desktop/Goritz_Lab/Spatial_transcriptomic/Analyse_3/For_RTCD/weights.csv", sep=";", index_col=0)
adata.obs = adata.obs.join(weights)

In [ ]:
## Plot specific cluster from leiden 0.5 (ie clustering based on non-spatial data)

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Select groups from metadata
groups = ['crush_d14_1']
mask = adata.obs['manual_anno'].isin(groups)

# Get Leiden cluster assignments for that subset
clusters = adata.obs.loc[mask, 'leiden_0.5'].astype(str)

# Clusters to highlight one by one
target_clusters = ["0", "1", "2", "3", "4", "5", "6"]

for cl in target_clusters:
    # Assign colors: red for the current cluster, grey otherwise
    colors = ["red" if label == cl else "lightgrey" for label in clusters]

    # Plot
    plt.figure(figsize=(35, 15))
    plt.scatter(
        adata.obs.loc[mask, 'x'],
        adata.obs.loc[mask, 'y'],
        c=colors,
        s=30,
        alpha=1
    )
    plt.axis("off")

    # Legend: just 2 entries
    handles = [
        mpatches.Patch(color="red", label=f"Leiden {cl}"),
        mpatches.Patch(color="lightgrey", label="Other")
    ]
    plt.legend(
        handles=handles,
        title="Leiden 0.5 clusters",
        bbox_to_anchor=(1.05, 1),
        loc="upper left",
        fontsize=20,
        title_fontsize=14
    )

    plt.title(f"Highlighted: Leiden {cl}", fontsize=24)
    plt.show()


In [ ]:
## Plot specific cluster from frist_type from RCTD deconvolution

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Select groups from metadata
groups = ['crush_d14_1']
mask = adata.obs['manual_anno'].isin(groups)

# Get cluster assignments for that subset
clusters = adata.obs.loc[mask, 'first_type'].astype(str)

# List of populations to highlight one by one
populations = [
    "Astrocytes",
    "Ependymal",
    "Fibro",
    "Endothelial",
    "Oligodendroglia",
    "Macrophages",
    "Microglia",
    "Neurons_V",
    "Neurons_D",
    "Pericytes"
]

for pop in populations:
    # Assign colors: red for the selected population, grey otherwise
    colors = ["red" if label == pop else "lightgrey" for label in clusters]

    # Plot
    plt.figure(figsize=(35, 15))
    plt.scatter(
        adata.obs.loc[mask, 'x'],
        adata.obs.loc[mask, 'y'],
        c=colors,
        s=30,
        alpha=1
    )
    plt.axis("off")

    # Legend: just 2 entries
    handles = [
        mpatches.Patch(color="red", label=pop),
        mpatches.Patch(color="lightgrey", label="Other")
    ]
    plt.legend(
        handles=handles,
        title="rtcd_clusters_first_type",
        bbox_to_anchor=(1.05, 1),
        loc="upper left",
        fontsize=20,
        title_fontsize=14
    )

    plt.title(f"Highlighted: {pop}", fontsize=24)
    plt.show()


In [ ]:
## Plot deconvolution results (weights) on tissue section per cell type

import matplotlib.pyplot as plt

# List of all weight columns
weight_columns = [
    'Oligodendroglia_rtcd_weights',
    'Macrophages_rtcd_weights',
    'Microglia_rtcd_weights',
    'Astrocytes_rtcd_weights',
    'Neurons_V_rtcd_weights',
    'Ependymal_rtcd_weights',
    'Neurons_D_rtcd_weights',
    'Pericytes_rtcd_weights',
    'Endothelial_cells_rtcd_weights',
    'Fibro_rtcd_weights'
]

# Select groups from metadata
groups = ['crush_d14_1']
mask = adata.obs['manual_anno'].isin(groups)

for col in weight_columns:
    # Convert to numeric (replace comma with dot, cast to float)
    values = (
        adata.obs.loc[mask, col]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .astype(float)
    )

    # Plot
    plt.figure(figsize=(35, 15))
    sc = plt.scatter(
        adata.obs.loc[mask, 'x'],
        adata.obs.loc[mask, 'y'],
        c=values,
        cmap="Reds",   # choose any colormap, e.g. "viridis"
        s=20,
        alpha=1
    )
    plt.axis("off")

    # Add colorbar
    cbar = plt.colorbar(sc)
    cbar.set_label(col, fontsize=14)

    # Title
    plt.title(col, fontsize=16)

    plt.show()


In [ ]:
## Plot average gene expression on tissue section

import matplotlib.pyplot as plt
import numpy as np

# Select groups from metadata
groups = ['crush_d14_1']
mask = adata.obs['manual_anno'].isin(groups)

# Gene list
gene_list = ["Col1a2", "Col3a1", "Col5a1", "Dcn", "Fn1", 'Col15a1', 'Col6a2', 'Col6a1']

# Make sure all genes are present
genes_present = [g for g in gene_list if g in adata.var_names]
print("Using genes:", genes_present)

# Extract expression (cells × genes subset)
expr = adata[mask, genes_present].X

# Convert to dense if sparse
if not isinstance(expr, np.ndarray):
    expr = expr.toarray()

# Compute average expression per cell
avg_expr = expr.mean(axis=1)

# Plot
plt.figure(figsize=(35, 15))
sc = plt.scatter(
    adata.obs.loc[mask, 'x'],
    adata.obs.loc[mask, 'y'],
    c=avg_expr,
    cmap="Reds",   # or "viridis"
    s=30,
    alpha=1
)
plt.axis("off")

# Add colorbar
cbar = plt.colorbar(sc)
cbar.set_label("Average expression", fontsize=16)

plt.title(f"Average expression of {', '.join(genes_present)}", fontsize=24)
plt.show()
